# Binary classification: movie reviews

The first real problem in the book. Multi-hot encoding, two Dense layers, and the moment the validation loss turns around while the training loss keeps falling.

**Runs on:** CPU — about 2 minutes &nbsp;·&nbsp; **Slides:** [Chapter 4 — Classification and Regression](../../../course-web-slides/ch04/index.html) &nbsp;·&nbsp; **Section:** 01 — Classifying movie reviews

---

## The data

In [ ]:
from keras.datasets import imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(
    num_words=10000
)

print(len(train_data), "training reviews")
print("first review, as word indices:", train_data[0][:12], "...")
print("label:", train_labels[0], " (1 = positive)")
print("largest index anywhere:", max(max(seq) for seq in train_data))

Expected output:

```
25000 training reviews
first review, as word indices: [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468] ...
label: 1  (1 = positive)
largest index anywhere: 9999
```

In [ ]:
word_index = imdb.get_word_index()
reverse_word_index = {v: k for k, v in word_index.items()}

# Indices 0, 1 and 2 are reserved for padding, start-of-sequence, and unknown.
decoded = " ".join(reverse_word_index.get(i - 3, "?") for i in train_data[0])
print(decoded[:400], "...")

## Multi-hot encoding

Lists of integers cannot be fed to a Dense layer. Turn each review into a 10,000-vector with a 1 at every index that appears — **word order is discarded entirely**, and chapter 14 will return to whether that matters.

In [ ]:
import numpy as np

def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension), dtype="float32")
    for i, sequence in enumerate(sequences):
        for j in sequence:
            results[i, j] = 1.
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
y_train = np.asarray(train_labels).astype("float32")
y_test = np.asarray(test_labels).astype("float32")

print(x_train.shape, x_train[0][:12])

## The model

In [ ]:
import keras
from keras import layers

model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="rmsprop",
              loss="binary_crossentropy",
              metrics=["accuracy"])

> **Note** — One unit, `sigmoid`, `binary_crossentropy`. That triple is fixed for binary classification, and chapter 20 puts all five such triples in one table.

## A validation set, held out by hand

In [ ]:
x_val, partial_x_train = x_train[:10000], x_train[10000:]
y_val, partial_y_train = y_train[:10000], y_train[10000:]

history = model.fit(partial_x_train, partial_y_train,
                    epochs=20, batch_size=512,
                    validation_data=(x_val, y_val), verbose=2)

## The plot this whole chapter exists for

In [ ]:
import matplotlib.pyplot as plt

h = history.history
epochs = range(1, len(h["loss"]) + 1)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(epochs, h["loss"], "o-", ms=3, label="training loss")
a1.plot(epochs, h["val_loss"], "s-", ms=3, label="validation loss")
a1.set_xlabel("epoch"); a1.legend(); a1.set_title("Loss")

a2.plot(epochs, h["accuracy"], "o-", ms=3, label="training accuracy")
a2.plot(epochs, h["val_accuracy"], "s-", ms=3, label="validation accuracy")
a2.set_xlabel("epoch"); a2.legend(); a2.set_title("Accuracy")
plt.tight_layout(); plt.show()

best = int(np.argmin(h["val_loss"])) + 1
print(f"validation loss is lowest at epoch {best}, then rises for "
      f"{len(epochs) - best} more epochs while training loss keeps falling")

**Training loss falls monotonically. Validation loss turns around at about epoch 4.** Everything after that point is the model memorising the training set. This picture is the reason chapter 5 exists.

## Retraining to the right number of epochs

In [ ]:
model = keras.Sequential([
    layers.Dense(16, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="rmsprop", loss="binary_crossentropy",
              metrics=["accuracy"])
model.fit(x_train, y_train, epochs=4, batch_size=512, verbose=0)
print("test:", model.evaluate(x_test, y_test, verbose=0))

Expected output:

```
test: [0.29xx, 0.88xx]
```

About 88%. A state-of-the-art model reaches roughly 95% — chapter 15 gets there with RoBERTa. Note how far a two-layer network on bag-of-words gets, and remember it when the expensive option is proposed.

---

## What to take away

- Sequences must be vectorized before a Dense layer will take them.
- Binary classification: one unit, sigmoid, binary_crossentropy.
- **Validation loss turning around while training loss falls is overfitting**, and it happens by epoch 4 here.
- A simple baseline reaches 88% on a problem where the ceiling is about 95%.